In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder , OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix

In [7]:
df = pd.read_csv("/content/loantrain.csv")

In [8]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [9]:
df.tail()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y
613,LP002990,Female,No,0,Graduate,Yes,4583,0.0,133.0,360.0,0.0,Semiurban,N


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [11]:
df.shape

(614, 13)

In [12]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [13]:
df.isnull().sum()

,0
Loan_ID,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14


In [14]:
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])

In [15]:
categorical_columns = [
    "Married",
    "Dependents",
    "Self_Employed",
    "Loan_Amount_Term",
    "Credit_History"
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [16]:
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].mean())

In [17]:
df.isnull().sum()

,0
Loan_ID,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,0


In [18]:
df["Dependents"] = df["Dependents"].replace("3+", "3")

In [19]:
df["Dependents"] = df["Dependents"].astype(int)

In [20]:
print(df["Dependents"].unique())
print(df["Dependents"].dtype)

[0 1 2 3]
int64


In [21]:
le = LabelEncoder()

In [22]:
encode_col = ["Gender","Married","Education","Self_Employed","Loan_Status"]

In [23]:
for col in encode_col:
  df[col] = le.fit_transform(df[col])

In [24]:
df = pd.get_dummies(
    df,
    columns=["Property_Area"],
    drop_first=True,
    dtype=int
)

In [25]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban
0,LP001002,1,0,0,0,0,5849,0.0,146.412162,360.0,1.0,1,0,1
1,LP001003,1,1,1,0,0,4583,1508.0,128.000000,360.0,1.0,0,0,0
2,LP001005,1,1,0,0,1,3000,0.0,66.000000,360.0,1.0,1,0,1
3,LP001006,1,1,0,1,0,2583,2358.0,120.000000,360.0,1.0,1,0,1
4,LP001008,1,0,0,0,0,6000,0.0,141.000000,360.0,1.0,1,0,1


In [26]:
X=df.drop(columns=["Loan_ID","Loan_Status"],axis=1)
Y=df["Loan_Status"]

In [27]:
X_train , X_test , Y_train , Y_test = train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=2)
print(X.shape,X_train.shape,X_test.shape)

(614, 12) (491, 12) (123, 12)


In [28]:
sc = StandardScaler()
X_train[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]] = sc.fit_transform(X_train[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]])
X_test[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]] = sc.transform(X_test[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]])

In [29]:
X.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area_Semiurban,Property_Area_Urban
0,1,0,0,0,0,5849,0.0,146.412162,360.0,1.0,0,1
1,1,1,1,0,0,4583,1508.0,128.000000,360.0,1.0,0,0
2,1,1,0,0,1,3000,0.0,66.000000,360.0,1.0,0,1
3,1,1,0,1,0,2583,2358.0,120.000000,360.0,1.0,0,1
4,1,0,0,0,0,6000,0.0,141.000000,360.0,1.0,0,1


In [30]:
models = [ LogisticRegression(max_iter=1000) , DecisionTreeClassifier() , RandomForestClassifier() ]

In [31]:
for model in models:
  model.fit(X_train,Y_train)

In [32]:
for model in models:
  y_pred = model.predict(X_test)
  print(model)
  print("Accuracy Score : " , accuracy_score(Y_test,y_pred))
  print("Confusion Matrix : " , confusion_matrix(Y_test,y_pred))
  print("Precision :", precision_score(Y_test,y_pred))
  print("Recall :", recall_score(Y_test,y_pred))
  print("F1 Score :", f1_score(Y_test,y_pred))
  print("------------------------------------------------------------")

LogisticRegression(max_iter=1000)
Accuracy Score :  0.8048780487804879
Confusion Matrix :  [[18 20]
 [ 4 81]]
Precision : 0.801980198019802
Recall : 0.9529411764705882
F1 Score : 0.8709677419354839
------------------------------------------------------------
DecisionTreeClassifier()
Accuracy Score :  0.7723577235772358
Confusion Matrix :  [[27 11]
 [17 68]]
Precision : 0.8607594936708861
Recall : 0.8
F1 Score : 0.8292682926829268
------------------------------------------------------------
RandomForestClassifier()
Accuracy Score :  0.7886178861788617
Confusion Matrix :  [[19 19]
 [ 7 78]]
Precision : 0.8041237113402062
Recall : 0.9176470588235294
F1 Score : 0.8571428571428571
------------------------------------------------------------


In [33]:
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ["l1", "l2"]

    },
    "Decision Tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 1, 2, 3, 4, 5],
        "min_samples_split": [2, 3, 4],
        "min_samples_leaf": [1, 2, 3]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }
}

In [34]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver="liblinear"),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}

In [35]:
#best_score = 0
#best_model = None
#best_params = None
#best_model_name = ""

#for name, model in models.items():
#    grid = GridSearchCV(estimator=model, param_grid = param_grids[name], cv=5, scoring = "accuracy")
#    grid.fit(X_train, Y_train)

#    print(f"{name}:")
#    print("Best Parameters :", grid.best_params_)
#    print(f"Best score for {name}: {grid.best_score_}")

#    if grid.best_score_ > best_score:
#        best_score = grid.best_score_
#        best_model = grid.best_estimator_
#        best_params = grid.best_params_
#        best_model_name = name
#print("\n==============================")
#print("Best Model :", best_model_name)
#print("Best Parameters :", best_params)
#print("Best Cross Validation Accuracy :", best_score)


In [36]:
#best_model.fit(X_train,Y_train)
#y_pred = best_model.predict(X_test)
#print("Accuracy Score : " , accuracy_score(Y_test,y_pred))
#print("Confusion Matrix :\n", confusion_matrix(Y_test, y_pred))
#print("Precision :", precision_score(Y_test, y_pred))
#print("Recall :", recall_score(Y_test, y_pred))
#print("F1 Score :", f1_score(Y_test, y_pred))

In [37]:
best_models = {}

for name, model in models.items():

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=5,
        scoring="accuracy"
    )

    grid.fit(X_train, Y_train)

    best_models[name] = grid.best_estimator_

    print(f"{name}:")
    print("Best Parameters :", grid.best_params_)
    print("Best CV Score :", grid.best_score_)
    print("-" * 60)

Logistic Regression:
Best Parameters : {'C': 0.1, 'penalty': 'l1'}
Best CV Score : 0.8105545248402392
------------------------------------------------------------
Decision Tree:
Best Parameters : {'criterion': 'gini', 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Score : 0.8125953411667698
------------------------------------------------------------
Random Forest:
Best Parameters : {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 100}
Best CV Score : 0.8105545248402392
------------------------------------------------------------


In [38]:
for name, model in best_models.items():

    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy :", accuracy_score(Y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_test, y_pred))
    print("Precision :", precision_score(Y_test, y_pred))
    print("Recall :", recall_score(Y_test, y_pred))
    print("F1 Score :", f1_score(Y_test, y_pred))
    print("-" * 60)


Logistic Regression
Accuracy : 0.8048780487804879
Confusion Matrix:
 [[17 21]
 [ 3 82]]
Precision : 0.7961165048543689
Recall : 0.9647058823529412
F1 Score : 0.8723404255319149
------------------------------------------------------------

Decision Tree
Accuracy : 0.8048780487804879
Confusion Matrix:
 [[18 20]
 [ 4 81]]
Precision : 0.801980198019802
Recall : 0.9529411764705882
F1 Score : 0.8709677419354839
------------------------------------------------------------

Random Forest
Accuracy : 0.8048780487804879
Confusion Matrix:
 [[17 21]
 [ 3 82]]
Precision : 0.7961165048543689
Recall : 0.9647058823529412
F1 Score : 0.8723404255319149
------------------------------------------------------------


In [39]:
best_accuracy = 0
final_model = None
final_model_name = ""

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(Y_test, y_pred)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        final_model = model
        final_model_name = name

print("Final Best Model :", final_model_name)
print("Final Test Accuracy :", best_accuracy)

Final Best Model : Logistic Regression
Final Test Accuracy : 0.8048780487804879


In [40]:


def predict_loan(
    gender,
    married,
    dependents,
    education,
    self_employed,
    applicant_income,
    coapplicant_income,
    loan_amount,
    loan_amount_term,
    credit_history,
    property_area
):

    # Convert Property Area into one-hot encoding
    property_area_semiurban = 1 if property_area == "Semiurban" else 0
    property_area_urban = 1 if property_area == "Urban" else 0

    # Create DataFrame
    data = pd.DataFrame({
        "Gender": [gender],
        "Married": [married],
        "Dependents": [dependents],
        "Education": [education],
        "Self_Employed": [self_employed],
        "ApplicantIncome": [applicant_income],
        "CoapplicantIncome": [coapplicant_income],
        "LoanAmount": [loan_amount],
        "Loan_Amount_Term": [loan_amount_term],
        "Credit_History": [credit_history],
        "Property_Area_Semiurban": [property_area_semiurban],
        "Property_Area_Urban": [property_area_urban]
    })

    # Scale numerical columns
    numerical_cols = [
        "ApplicantIncome",
        "CoapplicantIncome",
        "LoanAmount"
    ]

    data[numerical_cols] = sc.transform(data[numerical_cols])

    # Prediction
    prediction = final_model.predict(data)[0]

    if prediction == 1:
        return "Loan Approved"
    else:
        return "Loan Rejected"

In [41]:
predict_loan(
    gender=1,
    married=1,
    dependents=0,
    education=0,
    self_employed=0,
    applicant_income=5000,
    coapplicant_income=2000,
    loan_amount=120,
    loan_amount_term=360,
    credit_history=1,
    property_area="Urban"
)

'Loan Approved'

In [42]:
import pickle

In [46]:
filename = 'loan_model.sav'
with open("loan_model.sav", "wb") as file:
    pickle.dump(final_model, file)

In [47]:
with open("loan_model.sav", "rb") as file:
    loaded_model = pickle.load(file)

In [48]:
scaler_filename = "scaler.sav"
with open(scaler_filename, "wb") as file:
    pickle.dump(sc, file)
print("Model and Scaler saved successfully!")

Model and Scaler saved successfully!
